# Baseline forecasts on SIMULATED demand

This notebook builds next-day ward-level water demand forecasting baselines using SIMULATED data.
We compare a Naive lag-1 model, a Seasonal Naive lag-7 model, and an empirical `HistGradientBoostingRegressor`.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

# 1. Load simulated daily demand and ward metadata
sim = pd.read_csv('data/simulated_demand.csv', parse_dates=['date'])
wards = pd.read_csv('data/wards.csv')

# Merge static ward attributes
ward_cols = ['ward_id', 'base_mld', 'tanker_dependence', 'groundwater_stress', 'zone']
sim = sim.merge(wards[ward_cols], on='ward_id', how='left')

# Ensure deterministic order by ward and date
sim = sim.sort_values(by=['ward_id', 'date']).reset_index(drop=True)

print(f'Simulated demand records: {sim.shape}')
print(f'Wards metadata records: {wards.shape}')

Simulated demand records: (217008, 9)
Wards metadata records: (198, 9)


In [2]:
# 2. Feature engineering per ward using strictly past information
sim['day_of_week'] = sim['date'].dt.day_of_week
sim['month'] = sim['date'].dt.month

# Lags and rolling metrics computed per ward group
sim['lag_1'] = sim.groupby('ward_id')['demand_mld'].shift(1)
sim['lag_7'] = sim.groupby('ward_id')['demand_mld'].shift(7)
sim['rolling_7'] = sim.groupby('ward_id')['demand_mld'].transform(
    lambda s: s.shift(1).rolling(7).mean()
)

# Drop initial rows containing NaN values from lag calculation
df = sim.dropna().reset_index(drop=True)

features = [
    'day_of_week', 'month', 'rainfall', 'lag_1', 'lag_7', 'rolling_7',
    'base_mld', 'tanker_dependence', 'groundwater_stress', 'ward_id'
]
target = 'demand_mld'

print(f'Cleaned feature dataset shape (after dropping initial lag NaNs): {df.shape}')
print('Features list:')
print(features)

Cleaned feature dataset shape (after dropping initial lag NaNs): (215622, 14)
Features list:
['day_of_week', 'month', 'rainfall', 'lag_1', 'lag_7', 'rolling_7', 'base_mld', 'tanker_dependence', 'groundwater_stress', 'ward_id']


In [3]:
# 3. Temporal train / calibration / test split (60% / 20% / 20% by date)
unique_dates = np.sort(df['date'].unique())
n_dates = len(unique_dates)
n_train = int(n_dates * 0.60)
n_cal = int(n_dates * 0.20)

train_dates = unique_dates[:n_train]
cal_dates = unique_dates[n_train:n_train + n_cal]
test_dates = unique_dates[n_train + n_cal:]

train_df = df[df['date'].isin(train_dates)].reset_index(drop=True)
cal_df = df[df['date'].isin(cal_dates)].reset_index(drop=True)
test_df = df[df['date'].isin(test_dates)].reset_index(drop=True)

train_end = pd.to_datetime(train_dates[-1]).strftime('%Y-%m-%d')
cal_end = pd.to_datetime(cal_dates[-1]).strftime('%Y-%m-%d')

print('Split Breakdown (by Date):')
print(f'Train: {train_df["date"].min().strftime("%Y-%m-%d")} to {train_end} ({len(train_df):,} rows, {len(train_dates)} days)')
print(f'Cal:   {cal_df["date"].min().strftime("%Y-%m-%d")} to {cal_end} ({len(cal_df):,} rows, {len(cal_dates)} days)')
print(f'Test:  {test_df["date"].min().strftime("%Y-%m-%d")} to {test_df["date"].max().strftime("%Y-%m-%d")} ({len(test_df):,} rows, {len(test_dates)} days)')

# Persist split boundaries for downstream notebooks
split_dict = {'train_end': train_end, 'cal_end': cal_end}
with open('data/split_dates.json', 'w') as f:
    json.dump(split_dict, f, indent=2)
print(f'Saved data/split_dates.json: {split_dict}')

Split Breakdown (by Date):
Train: 2022-01-08 to 2023-10-22 (129,294 rows, 653 days)
Cal:   2023-10-23 to 2024-05-26 (42,966 rows, 217 days)
Test:  2024-05-27 to 2024-12-31 (43,362 rows, 219 days)
Saved data/split_dates.json: {'train_end': '2023-10-22', 'cal_end': '2024-05-26'}


In [4]:
# 4. Heuristic baselines on the test set
y_test = test_df[target].values
pred_naive = test_df['lag_1'].values
pred_s_naive = test_df['lag_7'].values

mae_naive = mean_absolute_error(y_test, pred_naive)
rmse_naive = np.sqrt(mean_squared_error(y_test, pred_naive))

mae_s_naive = mean_absolute_error(y_test, pred_s_naive)
rmse_s_naive = np.sqrt(mean_squared_error(y_test, pred_s_naive))

print(f'Naive baseline (lag_1):          MAE = {mae_naive:.3f} MLD | RMSE = {rmse_naive:.3f} MLD')
print(f'Seasonal Naive baseline (lag_7): MAE = {mae_s_naive:.3f} MLD | RMSE = {rmse_s_naive:.3f} MLD')

Naive baseline (lag_1):          MAE = 0.641 MLD | RMSE = 0.894 MLD
Seasonal Naive baseline (lag_7): MAE = 0.630 MLD | RMSE = 0.890 MLD


In [5]:
# 5. Train HistGradientBoostingRegressor on train set and evaluate
model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, random_state=42)
model.fit(train_df[features], train_df[target])

pred_model_test = model.predict(test_df[features])
mae_model = mean_absolute_error(y_test, pred_model_test)
rmse_model = np.sqrt(mean_squared_error(y_test, pred_model_test))

results = pd.DataFrame({
    'Model': ['Naive (lag_1)', 'Seasonal Naive (lag_7)', 'HistGradientBoosting'],
    'MAE (MLD)': [mae_naive, mae_s_naive, mae_model],
    'RMSE (MLD)': [rmse_naive, rmse_s_naive, rmse_model]
})

results.round(3)

,Model,MAE (MLD),RMSE (MLD)
0,Naive (lag_1),0.641,0.894
1,Seasonal Naive (lag_7),0.630,0.890
2,HistGradientBoosting,0.394,0.547


In [6]:
# 6. Extra views: Error by zone, Model Comparison bar chart, Permutation Importance
test_df_eval = test_df.copy()
test_df_eval['pred'] = pred_model_test
test_df_eval['abs_err'] = np.abs(test_df_eval[target] - test_df_eval['pred'])

zone_mae = test_df_eval.groupby('zone')['abs_err'].mean()
print('MAE per zone for HistGradientBoosting:')
print(zone_mae.apply(lambda v: f'{v:.3f} MLD'))

# Bar chart comparing the three MAEs
plt.figure(figsize=(7, 4))
bars = plt.bar(results['Model'], results['MAE (MLD)'], color=['grey', 'slategray', 'dodgerblue'], edgecolor='black')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0.015, f'{yval:.3f}', ha='center', va='bottom', fontsize=10)

plt.title('Test MAE (SIMULATED data)', fontsize=12, fontweight='bold')
plt.ylabel('MAE (MLD)')
plt.ylim(0, max(results['MAE (MLD)']) * 1.2)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Permutation importance on 20,000-row sample of test set
test_sample = test_df.sample(n=20000, random_state=42)
perm = permutation_importance(model, test_sample[features], test_sample[target], n_repeats=3, random_state=42)

sorted_idx = perm.importances_mean.argsort()
plt.figure(figsize=(8, 5))
plt.barh(np.array(features)[sorted_idx], perm.importances_mean[sorted_idx], color='teal', edgecolor='black')
plt.title('Feature Permutation Importance on Test Sample (SIMULATED)', fontsize=12, fontweight='bold')
plt.xlabel('Mean Importance Decrease in Score')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

MAE per zone for HistGradientBoosting:
zone
core         0.302 MLD
mid          0.373 MLD
periphery    0.428 MLD
Name: abs_err, dtype: object


In [7]:
# 7. Plot Actual vs Predicted for one periphery ward over first 90 days of test period
peri_ward = wards[wards['zone'] == 'periphery'].iloc[0]['ward_id']
peri_test = test_df_eval[test_df_eval['ward_id'] == peri_ward].sort_values('date').iloc[:90]

plt.figure(figsize=(12, 4))
plt.plot(peri_test['date'], peri_test['demand_mld'], label='Actual Demand', color='black', linewidth=1.5)
plt.plot(peri_test['date'], peri_test['pred'], label='HistGradientBoosting Prediction', color='crimson', linestyle='--', linewidth=1.5)
plt.title(f'SIMULATED Actual vs Predicted Demand: Periphery Ward {peri_ward} (First 90 Test Days)', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Demand (MLD)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [8]:
# 8. Save calibration and test predictions
pred_model_cal = model.predict(cal_df[features])

cal_preds = pd.DataFrame({
    'date': cal_df['date'],
    'ward_id': cal_df['ward_id'],
    'split': 'cal',
    'actual': cal_df[target],
    'pred': pred_model_cal
})

test_preds = pd.DataFrame({
    'date': test_df['date'],
    'ward_id': test_df['ward_id'],
    'split': 'test',
    'actual': test_df[target],
    'pred': pred_model_test
})

all_preds = pd.concat([cal_preds, test_preds], ignore_index=True)
all_preds.to_csv('data/baseline_preds.csv', index=False)
print(f'Saved data/baseline_preds.csv ({len(all_preds)} rows)')

Saved data/baseline_preds.csv (86328 rows)


## Methodological Note on Simulator Learning

> **Honest Evaluation Note:** The forecast errors reported here (e.g., test MAE of 0.394 MLD) measure how effectively the machine learning model reconstructs our underlying mathematical data-generating process (our simulator's seasonal, weekly, rainfall, and spatial shock mechanics). They do **not** reflect forecasting precision against real, metered Bengaluru municipal consumption, which contains human behavioral noise, unmetered tanker dispatching, and unmodeled infrastructure anomalies.